# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
import os
import sys
import pandas as pd

# enforce more deterministic behavior in cuBLAS operations.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# select a GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sys.path.append("..")

from processor.core.interaction_conductor.llm_conductor import LLMConductor
from processor.core.ir_system.lm_interface import LMInterface
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.core.ir_system.ir_data_model import AbstractDocument
from processor.core.ir_system.ir_data_model import Table, TableContext
from processor.core.ir_system.ir_data_model import Knowledge

In [ ]:
# llm_path = "model/weight/qwen3-1_7b"
llm_path = "gpt-4o-mini"
embed_model_path = "model/weight/bge-base"
logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=logging.INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
llm_conductor = LLMConductor(llm_path, embed_model_path, logger)

# IR Indexing

In [ ]:
DO_INDEXING_PNEUMA = False
DO_INDEXING_KB = False

In [ ]:
# ir_system = LMInterface({
#     "llm": llm_conductor.llm,
#     "embed_model": llm_conductor.embed_model
# }, llm_conductor.logger)

In [ ]:
# Pneuma indexing
if DO_INDEXING_PNEUMA:
    TABLES_PATH = "../../data_src/buysite"
    DATASET_NAME = "buysite"
    documents: list[AbstractDocument] = []
    metadata = pd.read_csv(f"{TABLES_PATH}/metadata.csv")
    for table_fname in os.listdir(f"{TABLES_PATH}/dataset"):
        try:
            table_name = f"{TABLES_PATH}/dataset/{table_fname}"
            table = pd.read_csv(table_name, nrows=100)
            documents.append(Table(
                doc_id=table_name,
                retriever_type=RetrieverType.PNEUMA,
                content=table,
                metadata={
                    "table_name": table_name,
                    "dataset_name": DATASET_NAME,
                }
            ))
            table_context = metadata[metadata["table"].str.endswith(table_fname[:-4].upper())].reset_index(drop=True)
            if len(table_context) > 0:
                documents.append(
                    TableContext(
                        doc_id=f"table_context_{table_name}",
                        retriever_type=RetrieverType.PNEUMA,
                        content=table_context["value"][0],
                        metadata={
                            "table_name": table_name,
                            "dataset_name": DATASET_NAME,
                            "type": "description",
                        }
                    )
                )
        except pd.errors.EmptyDataError:
            continue
    ir_system.index_documents(
        RetrieverType.PNEUMA,
        documents,
    )

In [ ]:
# Test Pneuma's retrieval
# x = ir_system.retrieve_documents(
#     "Which item is bought by vendor A in our dataset?", ["buysite"], 1
# )
# x[RetrieverType.PNEUMA][0].doc_id

In [ ]:
# KB indexing
if DO_INDEXING_KB:
    documents: list[AbstractDocument] = [
        Knowledge(
            doc_id="kb_1",
            retriever_type=RetrieverType.KNOWLEDGE_BASE,
            content="",
            metadata={
                "type": "local",
                "user": "James"
            }
        )
    ]
    ir_system.index_documents(
        RetrieverType.KNOWLEDGE_BASE, documents
    )

In [ ]:
# Test KB's retrieval
# x = ir_system.retrieve_documents(
#     "Does user wants local or global information?", [], 1
# )
# x[RetrieverType.KNOWLEDGE_BASE][0]

# E2E Evaluation

## Basic Evaluation

In [ ]:
QUESTION_1 = "I need to know if my team was preparing to receive shipping items on July 25, 2024. For reference, today is July 20, 2025."

In [ ]:
llm_conductor.process_input(QUESTION_1)

In [ ]:
llm_conductor.action_history

In [ ]:
llm_conductor.conversations

In [ ]:
llm_conductor.current_retrieval_results

In [ ]:
llm_conductor.state.get_state()

In [ ]:
x = "I understand you're looking for confirmation on whether your team was preparing to receive shipping items on July 25, 2024. Based on the current data we have, I will check for any relevant entries that indicate shipping preparations made before that date.\n\nThe target schema I have involves various fields related to shipping preparations, including the requested delivery date and creation timestamps to see if any items were recorded as prepared before July 25, 2024. Would you like me to proceed with querying the data, or is there any additional detail you'd like to specify?"
print(x)

# Testing Zone

In [ ]:
def convert_target_schemas_to_str(target_schemas: dict[str, dict[str, str]]):
    representation = ""
    for table_id, table in target_schemas.items():
        representation += f"- Table {table_id}:```\n"
        for col_name, col_desc in table.items():
            representation += f"-> col {col_name}: {col_desc}\n"
        representation += "```"
    return representation

print(convert_target_schemas_to_str(
    {
        "T1": {
            "col_1": "desc_1",
            "col_2": "desc_2"
        }
    }
))